<a href="https://colab.research.google.com/github/ipeirotis/dealing_with_data/blob/master/01-Pandas/B1_Reading_and_Writing_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B1: Reading and Writing Data — Multiple Sources

## Beyond BigQuery

In notebooks A1-A4, we loaded data exclusively from Google BigQuery. In the real world, data comes in many formats:

| Format | Use Case | Pandas Function |
|--------|----------|----------------|
| **CSV** | Universal data exchange | `read_csv()` |
| **Excel** | Business reports, spreadsheets | `read_excel()` |
| **SQL Database** | Production systems, data warehouses | `read_sql()` |
| **HTML Tables** | Web scraping | `read_html()` |
| **JSON** | APIs, web services | `read_json()` |
| **Fixed-Width** | Legacy systems, government data | `read_fwf()` |

This notebook covers how to:
1. **Read** data from various sources into Pandas DataFrames
2. **Write** data back to files and databases
3. **Reshape** data between wide and long formats
4. **Choose** the right approach for performance

## Learning Objectives

By completing this notebook, you will be able to:
- Load data from CSV, Excel, SQL databases, and web pages
- Export DataFrames to various formats
- Use `melt()` to reshape data from wide to long format
- Understand when to process data in SQL vs. Pandas

---

## Setup

In [ ]:
# Install required libraries
!pip install -q PyMySQL sqlalchemy xlrd openpyxl lxml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = [10, 4]

print("✓ Setup complete!")

---

## Part 1: Reading CSV Files

CSV (Comma-Separated Values) is the most common format for data exchange. Pandas can read CSVs from:
- Local files
- URLs
- Cloud storage

### Basic CSV Reading

In [ ]:
# Option 1: Read directly from a URL
url = 'https://data.cityofnewyork.us/api/views/vfnx-vebw/rows.csv?accessType=DOWNLOAD'
squirrels = pd.read_csv(url)

print(f"Loaded {len(squirrels):,} rows")
squirrels.head()

In [ ]:
# Option 2: Download first, then read (better for large files you'll reload)
!curl -s 'https://data.cityofnewyork.us/api/views/vfnx-vebw/rows.csv?accessType=DOWNLOAD' -o squirrel_census.csv

squirrels = pd.read_csv('squirrel_census.csv')
squirrels.head()

### Common `read_csv()` Parameters

| Parameter | Purpose | Example |
|-----------|---------|--------|
| `sep` | Delimiter (default: comma) | `sep='\t'` for TSV |
| `header` | Row number for column names | `header=0` (default) |
| `names` | Custom column names | `names=['a', 'b', 'c']` |
| `usecols` | Read only specific columns | `usecols=['col1', 'col2']` |
| `dtype` | Specify data types | `dtype={'zip': str}` |
| `parse_dates` | Parse date columns | `parse_dates=['date']` |
| `na_values` | Additional NA markers | `na_values=['N/A', '']` |
| `nrows` | Read only first n rows | `nrows=1000` |

In [ ]:
# Example: Reading with specific options
squirrels = pd.read_csv(
    'squirrel_census.csv',
    usecols=['X', 'Y', 'Unique Squirrel ID', 'Primary Fur Color', 'Age'],
    na_values=['?', '']  # Treat these as missing
)
squirrels.head()

---

## Part 2: Reading Excel Files

Excel files are common in business settings. Pandas can read `.xls` and `.xlsx` files.

### Basic Excel Reading

In [ ]:
# Read an Excel file from URL
titanic_url = 'https://storage.googleapis.com/datasets_nyu/titanic.xls'
titanic = pd.read_excel(titanic_url)

print(f"Loaded {len(titanic):,} passengers")
titanic.head()

### Reading Specific Sheets

Excel files often have multiple sheets. Use `sheet_name` to specify which one:

In [ ]:
# Example: NYC Restaurant Inspection Data Dictionary
url = 'https://data.cityofnewyork.us/api/views/43nn-pn8j/files/ec33d2c8-81f5-499a-a238-0213a38239cd?download=true&filename=RestaurantInspectionDataDictionary_09242018.xlsx'

# Read second sheet (index 1), use row 1 as header
data_dictionary = pd.read_excel(url, sheet_name=1, header=1)
data_dictionary.head()

In [ ]:
# Clean up the column names
data_dictionary.columns = ['Column_Name', 'Description', 'Code_Definitions', 'Notes']
data_dictionary = data_dictionary.dropna(subset=['Column_Name'])  # Remove empty rows
data_dictionary.head(10)

---

## Part 3: Reading from SQL Databases

For production data, you'll often connect to SQL databases (MySQL, PostgreSQL, SQLite, etc.).

### Connecting to MySQL

In [ ]:
from sqlalchemy import create_engine, text

# Create connection string
conn_string = 'mysql+pymysql://{user}:{password}@{host}/{db}?charset=utf8mb4'.format(
    host='db.ipeirotis.org',
    user='student',
    password='dwdstudent2015',
    db='imdb'
)

engine = create_engine(conn_string)
print("✓ Connected to MySQL database")

In [ ]:
# Read data using SQL query
query = '''
SELECT * FROM actors LIMIT 10
'''

with engine.connect() as connection:
    df_actors = pd.read_sql(text(query), con=connection)

df_actors

### SQL vs. Pandas: Performance Considerations

When working with large datasets, **where** you do the computation matters:

In [ ]:
%%time
# SLOW: Fetch all data, then aggregate in Pandas
query = '''SELECT * FROM movies'''

with engine.connect() as conn:
    df_all = pd.read_sql(text(query), con=conn)

# Aggregate in Pandas
result_pandas = df_all.groupby('year').agg(
    all_movies=('id', 'count'),
    rated_movies=('rating', 'count')
)

print(f"Fetched {len(df_all):,} rows, aggregated to {len(result_pandas)} rows")

In [ ]:
%%time
# FAST: Aggregate in SQL, fetch only the result
query = '''
SELECT year, COUNT(*) AS all_movies, COUNT(rating) AS rated_movies
FROM movies
GROUP BY year
ORDER BY year
'''

with engine.connect() as conn:
    result_sql = pd.read_sql(text(query), con=conn)

print(f"Fetched {len(result_sql)} rows directly")

**Rule of thumb**:
- Do filtering (`WHERE`) and aggregation (`GROUP BY`) in SQL
- Do complex transformations and visualization in Pandas

---

## Part 4: Reading Tables from Web Pages

Pandas can scrape HTML tables directly from web pages using `read_html()`.

### Example: Country Population from Wikipedia

In [ ]:
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)'

# read_html returns a LIST of all tables found on the page
df_list = pd.read_html(
    url,
    match='Population',  # Only tables containing this text
    header=0,
    storage_options={'User-Agent': 'Mozilla/5.0'} # Avoid Wikipedia block
)

print(f"Found {len(df_list)} matching table(s)")

In [ ]:
# Get the first (and usually main) table
df_population = df_list[0]
df_population.head(10)

In [ ]:
# Clean up: Remove Wikipedia footnotes like [4]
df_population = df_population.replace(
    to_replace=r'(.*)\[.\]',  # Regex pattern
    value=r'\1',
    regex=True
)

# Keep only useful columns
df_population.columns = ['Country', 'Pop_2022', 'Pop_2023', 'Change', 'Region', 'Subregion']
df_population = df_population[['Country', 'Pop_2023']]
df_population.head(10)

---

## Part 5: Reading Fixed-Width Files

Some legacy systems and government data use fixed-width format (columns at specific character positions).

In [ ]:
# Example: Monthly accidental deaths in the USA
deaths = pd.read_fwf('https://storage.googleapis.com/datasets_nyu/acc-deaths.txt')
display(deaths)

---

## Part 6: Reshaping Data with `melt()`

The deaths data above is in **wide format** (months as columns). Often we need **long format** (one row per observation).

| Wide Format | Long Format |
|-------------|-------------|
| Year, Jan, Feb, Mar, ... | Year, Month, Deaths |
| 1973, 9007, 8106, ... | 1973, Jan, 9007 |
| | 1973, Feb, 8106 |

### `melt()` converts wide → long
### `pivot_table()` converts long → wide

In [ ]:
# Convert from wide to long format using melt()
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

deaths_long = pd.melt(
    deaths,
    id_vars=['Year'],        # Columns to keep as-is
    value_vars=months,       # Columns to unpivot
    var_name='Month',        # Name for the new column holding old column names
    value_name='Deaths'      # Name for the new column holding values
)

deaths_long.head(15)

In [ ]:
# Create a proper date column
deaths_long['Date'] = pd.to_datetime(
    deaths_long['Month'] + '-' + deaths_long['Year'].astype(str),
    format='%b-%Y'
)

# Clean up and sort
deaths_long = (deaths_long
    .drop(['Month', 'Year'], axis='columns')
    .set_index('Date')
    .sort_index()
)

deaths_long.head()

In [ ]:
# Now we can plot as a time series!
deaths_long.plot(figsize=(8, 2), title='Monthly Accidental Deaths in the USA')
plt.ylabel('Deaths')

In [ ]:
# Convert back to wide format using pivot_table()
deaths_wide = deaths_long.reset_index()
deaths_wide['Year'] = deaths_wide['Date'].dt.year
deaths_wide['Month'] = deaths_wide['Date'].dt.strftime('%b')

deaths_wide.pivot_table(
    index='Year',
    columns='Month',
    values='Deaths'
).head()

---

## Part 7: Writing Data

Pandas can export DataFrames to many formats.

### Writing to CSV and Excel

In [ ]:
# Create a sample DataFrame
sample = titanic.head(100)

# Write to CSV
sample.to_csv('titanic_sample.csv', index=False)
print("✓ Saved to titanic_sample.csv")

# Write to Excel
sample.to_excel('titanic_sample.xlsx', index=False)
print("✓ Saved to titanic_sample.xlsx")

### Writing to SQL Database

In [ ]:
# Example: Writing to a database (connection setup)
# Note: This requires write permissions, which student accounts typically don't have

# conn_string = 'mysql+pymysql://{user}:{password}@{host}/{db}'.format(
#     host='your-server',
#     user='your-username',
#     password='your-password',
#     db='your-database'
# )
# engine = create_engine(conn_string)

# Write DataFrame to SQL table
# df.to_sql(
#     name='table_name',
#     con=engine,
#     if_exists='replace',  # or 'append' or 'fail'
#     index=False
# )



---

## 🎯 Practice Activities

### Activity 1: Load and Explore a CSV Dataset

Load the NYC 311 Service Requests dataset (sample) and explore it:
- URL: `https://data.cityofnewyork.us/api/views/erm2-nwe9/rows.csv?accessType=DOWNLOAD&$limit=10000`
- Find how many rows and columns
- Display the data types
- Count complaints by borough

In [ ]:
# YOUR CODE HERE


### Activity 2: Scrape a Wikipedia Table

Get the list of countries by life expectancy from Wikipedia:
- URL: `https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy`
- Extract the WHO data table
- Create a clean DataFrame with Country and Life Expectancy columns

In [ ]:
# YOUR CODE HERE


### Activity 3: Reshape Wide Data

The deaths dataset is in wide format. Use `melt()` to convert it to long format, then:
- Calculate the average deaths per month (across all years)
- Plot a bar chart showing seasonal patterns

In [ ]:
# YOUR CODE HERE


---

## 📝 Solutions

In [ ]:
# =============================================================================
# SOLUTION: Activity 1 - Load and Explore CSV
# =============================================================================

url = 'https://data.cityofnewyork.us/resource/erm2-nwe9.csv?$limit=10000'
complaints = pd.read_csv(url)

print(f"Shape: {complaints.shape[0]:,} rows × {complaints.shape[1]} columns")
print(f"\nData types:\n{complaints.dtypes}")
print(f"\nComplaints by Borough:")
print(complaints['borough'].value_counts())

In [ ]:
# =============================================================================
# SOLUTION: Activity 2 - Scrape Wikipedia Table
# =============================================================================

url = 'https://en.wikipedia.org/wiki/List_of_countries_by_life_expectancy'
tables = pd.read_html(url, match='Life expectancy at birth', header=0, storage_options={'User-Agent': 'Mozilla/5.0'})

df_life = tables[0]
print(f"Found table with {len(df_life)} rows")

display(df_life.head(3))

# Clean up
df_life = df_life[['Countries', 'Life expectancy at birth']].copy()
df_life.columns = ['Country', 'Life_Expectancy']
df_life = df_life.dropna()
df_life['Life_Expectancy'] = pd.to_numeric(df_life['Life_Expectancy'], errors='coerce')

print(f"\nTop 10 countries by life expectancy:")
df_life.sort_values('Life_Expectancy', ascending=False).head(10)

In [ ]:
# =============================================================================
# SOLUTION: Activity 3 - Reshape and Analyze
# =============================================================================

# Reload deaths data
deaths = pd.read_fwf('https://storage.googleapis.com/datasets_nyu/acc-deaths.txt')

# Melt to long format
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

deaths_long = pd.melt(
    deaths,
    id_vars=['Year'],
    value_vars=months,
    var_name='Month',
    value_name='Deaths'
)

# Make Month an ordered categorical for proper sorting
deaths_long['Month'] = pd.Categorical(deaths_long['Month'], categories=months, ordered=True)

# Calculate average by month
monthly_avg = deaths_long.groupby('Month', observed=False)['Deaths'].mean()

# Plot
monthly_avg.plot(kind='bar', figsize=(8, 3), color='steelblue')
plt.title('Average Monthly Accidental Deaths (Seasonal Pattern)')
plt.ylabel('Average Deaths')
plt.xlabel('Month')
plt.xticks(rotation=45)
plt.tight_layout()

print("Note: Higher deaths in summer months (outdoor activities) and winter (driving conditions)")

---

## Summary: Quick Reference

### Reading Data

```python
# CSV
df = pd.read_csv('file.csv')
df = pd.read_csv(url)

# Excel
df = pd.read_excel('file.xlsx', sheet_name=0)

# SQL
from sqlalchemy import create_engine, text
engine = create_engine(conn_string)
with engine.connect() as conn:
    df = pd.read_sql(text(query), con=conn)

# HTML Tables
tables = pd.read_html(url, match='pattern')
df = tables[0]

# Fixed-Width
df = pd.read_fwf('file.txt')
```

### Writing Data

```python
df.to_csv('output.csv', index=False)
df.to_excel('output.xlsx', index=False)
df.to_sql('table_name', con=engine, if_exists='replace')
```

### Reshaping

```python
# Wide → Long
df_long = pd.melt(df, id_vars=['keep'], value_vars=['a', 'b'])

# Long → Wide  
df_wide = df.pivot_table(index='row', columns='col', values='val')
```

### Performance Rule

**Do filtering and aggregation in SQL when possible. Fetch only the data you need.**